# Getting Started with TIP & PATSTAT

**Purpose:** Simple starter queries to learn how to search patents on the EPO Technology Intelligence Platform

**Target audience:** Patent information professionals with no SQL experience

**Platform:** EPO Technology Intelligence Platform (TIP)

**Edition:** PATSTAT Global, Autumn 2025

**Created:** 2026-03-02 | mtc.berlin for EPO Academy

---

**What you will learn:**
1. How to connect to PATSTAT and run a query
2. Search patents by applicant name
3. Filter by jurisdiction (filing authority)
4. Look up patent families for an applicant


## Setup: Connect to PATSTAT

In [9]:
from epo.tipdata.patstat import PatstatClient
import pandas as pd
import time

# Connect to PATSTAT
patstat = PatstatClient(env='PROD')

def run_query(query):
    """Execute a PATSTAT SQL query and return a pandas DataFrame."""
    start = time.time()
    res = patstat.sql_query(query, use_legacy_sql=False)
    df = pd.DataFrame(res)
    print(f"Query took {time.time() - start:.1f}s - {len(df)} rows returned")
    return df
print("Ready! Proceed to next cell.")

Ready! Proceed to next cell.


---

## Query 1: Find Patents by Applicant Name

Search for all patent applications from a specific company.

**Try it:** Change `'%siemens%'` to any company you are interested in (e.g. `'%bosch%'`, `'%samsung%'`, `'%basf%'`).

In [11]:
# --- CHANGE THIS ---
APPLICANT = '%siemens%'
# -------------------

df_q1 = run_query(f"""
SELECT
    a.appln_auth AS authority,
    a.appln_nr_epodoc AS application_number,
    a.appln_filing_date AS filing_date,
    a.appln_filing_year AS filing_year,
    a.granted,
    p.person_name AS applicant_name,
    p.person_ctry_code AS applicant_country
FROM tls201_appln a
JOIN tls207_pers_appln pa ON a.appln_id = pa.appln_id
JOIN tls206_person p ON pa.person_id = p.person_id
WHERE pa.applt_seq_nr > 0
  AND LOWER(p.person_name) LIKE '{APPLICANT}'
  AND a.appln_filing_year BETWEEN 2020 AND 2024
ORDER BY a.appln_filing_date DESC
LIMIT 100
""")

df_q1

Query took 0.5s - 100 rows returned


,authority,application_number,filing_date,filing_year,granted,applicant_name,applicant_country
0,WO,WO2024EP88686,2024-12-31,2024,N,SIEMENS ENERGY GLOBAL GMBH & CO. KG,DE
1,CN,CN202411997043,2024-12-31,2024,N,BEIJING SIEMENS CERBERUS ELECTRONICS LTD.,
2,CN,CN202411998275,2024-12-31,2024,N,SIEMENS SHANGHAI MEDICAL EQUIPMENT LTD.,
3,WO,WO2024EP88681,2024-12-31,2024,N,Siemens Aktiengesellschaft,DE
4,CN,CN202411986066,2024-12-31,2024,N,SIEMENS LTD. CHINA,
...,...,...,...,...,...,...,...
95,US,US202418973173,2024-12-09,2024,N,Siemens Healthineers AG,DE
96,WO,WO2024EP85274,2024-12-09,2024,N,Siemens Aktiengesellschaft,DE
97,WO,WO2024EP85265,2024-12-09,2024,N,Siemens Aktiengesellschaft,DE
98,US,US202418973494,2024-12-09,2024,N,Siemens Healthcare Diagnostics Inc.,US


---

## Query 2: Filter by Jurisdiction

Find patents from an applicant filed at specific patent offices.

**Try it:** Change the `AUTHORITIES` list to the jurisdictions you need (e.g. `('EP', 'US')` or `('CN', 'JP', 'KR')`).

In [6]:
# --- CHANGE THESE ---
APPLICANT = '%siemens%'
AUTHORITIES = "('EP', 'US', 'CN')"  # one or more: EP, US, CN, DE, JP, KR, WO ...
# --------------------

df_q2 = run_query(f"""
SELECT
    a.appln_auth AS authority,
    COUNT(*) AS filings
FROM tls201_appln a
JOIN tls207_pers_appln pa ON a.appln_id = pa.appln_id
JOIN tls206_person p ON pa.person_id = p.person_id
WHERE pa.applt_seq_nr > 0
  AND LOWER(p.person_name) LIKE '{APPLICANT}'
  AND a.appln_auth IN {AUTHORITIES}
  AND a.appln_filing_year BETWEEN 2020 AND 2024
GROUP BY a.appln_auth
ORDER BY filings DESC
""")

df_q2

Query took 2.0s - 3 rows returned


,authority,filings
0,EP,8798
1,CN,7378
2,US,6704


---

## Query 3: Detailed Filings per Jurisdiction and Year

See how many patents were filed per year at each selected office. Useful to spot trends.

In [7]:
# --- CHANGE THESE ---
APPLICANT = '%siemens%'
AUTHORITIES = "('EP', 'US', 'CN')" 
# --------------------

df_q3 = run_query(f"""
SELECT
    a.appln_auth AS authority,
    a.appln_filing_year AS filing_year,
    COUNT(*) AS filings
FROM tls201_appln a
JOIN tls207_pers_appln pa ON a.appln_id = pa.appln_id
JOIN tls206_person p ON pa.person_id = p.person_id
WHERE pa.applt_seq_nr > 0
  AND LOWER(p.person_name) LIKE '{APPLICANT}'
  AND a.appln_auth IN {AUTHORITIES}
  AND a.appln_filing_year BETWEEN 2015 AND 2024
GROUP BY a.appln_auth, a.appln_filing_year
ORDER BY a.appln_auth, a.appln_filing_year
""")

# Pivot into a readable table: years as rows, authorities as columns
df_q3_pivot = df_q3.pivot(index='filing_year', columns='authority', values='filings').fillna(0).astype(int)
df_q3_pivot

Query took 6.2s - 30 rows returned


authority,CN,EP,US
filing_year,,,
2015,1181,1842,1991
2016,1079,1666,1590
2017,1159,2177,1789
2018,1166,2604,1914
2019,1432,2576,1894
2020,1537,2214,1842
2021,1715,2249,1685
2022,1828,2117,1577
2023,1590,1914,1017


---

## Query 4: Patent Families for an Applicant

A patent family groups all filings worldwide that protect the **same invention**. This query shows how broadly an applicant protects its inventions.

Each row is one DOCDB family with:
- The earliest filing date (priority date)
- How many countries it was filed in (family size)
- Which authorities received filings

In [8]:
# --- CHANGE THIS ---
APPLICANT = '%siemens%'
# -------------------

df_q4 = run_query(f"""
SELECT
    a.docdb_family_id,
    MIN(a.appln_filing_date) AS earliest_filing,
    a.docdb_family_size AS family_size,
    COUNT(DISTINCT a.appln_auth) AS nr_authorities,
    STRING_AGG(DISTINCT a.appln_auth, ', ' ORDER BY a.appln_auth) AS authorities
FROM tls201_appln a
JOIN tls207_pers_appln pa ON a.appln_id = pa.appln_id
JOIN tls206_person p ON pa.person_id = p.person_id
WHERE pa.applt_seq_nr > 0
  AND LOWER(p.person_name) LIKE '{APPLICANT}'
  AND a.appln_filing_year BETWEEN 2020 AND 2024
  AND a.docdb_family_id > 0
GROUP BY a.docdb_family_id, a.docdb_family_size
ORDER BY family_size DESC
LIMIT 100
""")

df_q4

Query took 7.4s - 100 rows returned


,docdb_family_id,earliest_filing,family_size,nr_authorities,authorities
0,55696924,2020-11-13,25,1,US
1,80683772,2022-03-15,24,4,"BR, CN, EP, US"
2,56564560,2020-09-25,18,2,"JP, US"
3,54324481,2020-02-05,16,1,US
4,26010138,2020-02-10,15,1,US
...,...,...,...,...,...
95,45370793,2020-08-13,9,1,US
96,77042779,2021-07-23,9,8,"AU, BR, CN, EP, KR, PL, US, WO"
97,70475974,2020-04-28,9,6,"DK, EP, KR, PL, US, WO"
98,69846394,2020-03-03,9,7,"CN, DK, EP, ES, HR, KR, WO"


---

## Query 5: Patent Families Filtered by Jurisdiction

Same as above, but only showing families that include filings in your selected jurisdictions.

**Example use case:** "Show me all Siemens inventions that were filed at both the EPO and in China."

In [ ]:
# --- CHANGE THESE ---
APPLICANT = '%siemens%'
MUST_INCLUDE_1 = 'EP'   # families must include this authority
MUST_INCLUDE_2 = 'CN'   # AND this authority
# --------------------

df_q5 = run_query(f"""
WITH family_auths AS (
    SELECT
        a.docdb_family_id,
        MIN(a.appln_filing_date) AS earliest_filing,
        MAX(a.docdb_family_size) AS family_size,
        COUNT(DISTINCT a.appln_auth) AS nr_authorities,
        STRING_AGG(DISTINCT a.appln_auth, ', ' ORDER BY a.appln_auth) AS authorities
    FROM tls201_appln a
    JOIN tls207_pers_appln pa ON a.appln_id = pa.appln_id
    JOIN tls206_person p ON pa.person_id = p.person_id
    WHERE pa.applt_seq_nr > 0
      AND LOWER(p.person_name) LIKE '{APPLICANT}'
      AND a.appln_filing_year BETWEEN 2020 AND 2024
      AND a.docdb_family_id > 0
    GROUP BY a.docdb_family_id
    HAVING
      COUNTIF(a.appln_auth = '{MUST_INCLUDE_1}') > 0
      AND COUNTIF(a.appln_auth = '{MUST_INCLUDE_2}') > 0
)
SELECT *
FROM family_auths
ORDER BY family_size DESC
LIMIT 100
""")

df_q5

---

## What's Next?

Now that you can search by applicant and jurisdiction, explore further:

- **`0_airbus_filing_strategy_analysis.ipynb`** - Full 10-query strategy analysis for Airbus
- **`2_tu_dortmund_patent_portfolio.ipynb`** - University patent portfolio analysis

### Tips

- Use `han_name` instead of `person_name` for cleaner applicant matching (harmonized by EPO)
- Always add `applt_seq_nr > 0` to filter for applicants (not inventors)
- Use `docdb_family_id` to count inventions instead of individual filings
- Add `AND a.appln_kind = 'A'` to exclude utility models and design patents